# E7 — explicit field-presence control

**One variable on the E5 recipe:** six control tokens `<P:0/1/?>` `<Q:0/1/?>`,
with control dropout 0.5, so one model learns both the conditional and the
unconditional mode. Registered in `research/e7/PREREG_E7.md` before this run.

**Attach TWO datasets** (right panel → *Add Input*):

| dataset | what it is | how the notebook finds it |
|---|---|---|
| `uie-corpus2` | the data bundle, **unchanged** | `bundle.manifest.json` |
| `uie-e7-code` | the E7 `uie` package | `E7_CODE_MANIFEST.json` |

**Settings → Internet → On is required:** `tldextract` is not in the Kaggle
image, and cell 2 installs it before anything imports `uie`.

The data bundle carries its own **older** `uie`. Only the code dataset goes on
`sys.path`, and cell 2 proves which code is running before any GPU time is spent.

Lessons carried from `notebooks/kaggle_train.ipynb` and the R2 run: the mount
paths are **discovered**, never hard-coded, and training runs **in-process**.

In [ ]:
# --- 1. locate both datasets ---------------------------------------------------
from pathlib import Path

root = Path("/kaggle/input")
data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
code_hits = sorted(root.rglob("E7_CODE_MANIFEST.json")) if root.exists() else []

if not data_hits or not code_hits:
    print("tree under /kaggle/input:")
    for p in (sorted(root.rglob("*"))[:80] if root.exists() else []):
        print("   ", p)
    raise SystemExit(
        "Attach BOTH datasets (right panel -> Add Input):\n"
        "  uie-corpus2  -- the data, has bundle.manifest.json\n"
        "  uie-e7-code  -- the E7 code, has E7_CODE_MANIFEST.json\n"
        "Inputs mount when the session starts, so re-run after attaching.")

DATA = data_hits[0].parent
CODE = code_hits[0].parent
print("data:", DATA)
print("code:", CODE)

In [ ]:
# --- 2. environment: install first, then run the E7 code and PROVE it --------
import hashlib, importlib, json, subprocess, sys
from importlib import metadata

# tldextract FIRST. It is not in the Kaggle image, and `uie.tokenizer` imports it
# (through uie.factorization), so it must exist before any uie import -- attempt 1
# died here for exactly that reason. Pinned to the version the local evaluation
# parses with: its offline Public Suffix List decides how every host splits into
# fields. Checked through package metadata, so no other copy is ever imported.
TLDEXTRACT = "5.3.1"


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: "
                         "Settings -> Internet -> On, then rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")
print("tldextract:", TLDEXTRACT)

# CODE ONLY on the path. The data bundle's own `uie` is older and has no E7
# controls; if it shadowed this one the run would train a plain E5 model and
# look perfectly healthy while doing it.
sys.path.insert(0, str(CODE))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")]:
    del sys.modules[name]

import uie
from uie.tokenizer import URLTokenizer

manifest = json.loads((CODE / "E7_CODE_MANIFEST.json").read_text())
print("uie imported from:", uie.__file__)
assert str(CODE) in uie.__file__, "the E7 code is SHADOWED -- check sys.path"
assert URLTokenizer(include_fields=True).include_fields, "this uie has no E7 controls"

bad = [f for f, h in manifest["files"].items()
       if hashlib.sha256((CODE / f).read_bytes()).hexdigest() != h]
print(f"code integrity: {len(manifest['files']) - len(bad)}/{len(manifest['files'])} "
      f"files match the manifest")
assert not bad, f"files differ from the manifest: {bad}"
print(f"E7 code: commit {manifest['commit'][:12]} on {manifest['branch']}")

import torch
print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")
print("gpu  :", torch.cuda.get_device_name(0))

In [ ]:
# --- 3. the data bundle is the two-corpus one --------------------------------
need = ["name_train", "structure_train", "phishing_name_train",
        "phishing_structure_train", "name_val", "structure_val",
        "phishing_name_val", "phishing_structure_val"]
present = {p.name for p in DATA.iterdir()}
missing = [s for s in need
           if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
for s in need:
    print(("OK   " if s not in missing else "MISS "), s)
if missing:
    raise SystemExit(f"STOP. missing {missing}: this is not the uie-corpus2 bundle.")
print("\nready")

In [ ]:
# --- 4. train -----------------------------------------------------------------
# EVERY ARGUMENT IS STATED. Everything except the last two is E5's recipe;
# run()'s own defaults (d_model 256, 6 layers, no frame) would silently train a
# different model.
#
# Validation and best.pt selection run UNCONDITIONAL (controls shown as `?`),
# the same task and targets as E5's validation loss, so the two stay comparable.
#
# Watch the log for the line "E7: presence controls ON" and "vocab: 107".
# ~1.1 h expected on a T4 (R2's 40,000 steps took 7,461 s).
from uie.kaggle_train import run

result = run(
    DATA,
    Path("/kaggle/working"),
    steps=20000,          # E5
    batch_size=256,       # E5
    lr=3e-4,              # E5
    d_model=384,          # E5
    n_layers=8,           # E5
    n_heads=8,            # E5
    eval_every=1000,
    class_alpha=1.0,      # E5
    frame=True,           # E5
    factorized=False,     # E6 was rejected
    boundary_weight=1.0,  # S4 stays closed
    fields=True,          # E7: <P:0/1/?> <Q:0/1/?>
    field_dropout=0.5,    # E7: registered
)
print("\ndone")

In [ ]:
# --- 5. read the config back from the CHECKPOINT, not from the call site -----
import json
import torch

ck = torch.load("/kaggle/working/best.pt", map_location="cpu", weights_only=False)
tc, mc, tk = ck["train_config"], ck["model_config"], ck["tokenizer"]

checks = [
    ("steps", tc.get("steps"), 20000), ("batch_size", tc.get("batch_size"), 256),
    ("lr", tc.get("lr"), 3e-4), ("class_alpha", tc.get("class_alpha"), 1.0),
    ("boundary_weight", tc.get("boundary_weight"), 1.0),
    ("field_dropout", tc.get("field_dropout"), 0.5),
    ("seed", tc.get("seed"), 20260803),
    ("d_model", mc.get("d_model"), 384), ("n_layers", mc.get("n_layers"), 8),
    ("n_heads", mc.get("n_heads"), 8), ("max_len", mc.get("max_len"), 256),
    ("vocab_size", mc.get("vocab_size"), 107),
    ("include_frame", tk.get("include_frame"), True),
    ("include_fields", tk.get("include_fields"), True),
]
wrong = [c for c in checks if c[1] != c[2]]
for name, got, want in checks:
    print(f"{'OK   ' if got == want else 'WRONG'} {name:<16} expected {want}  got {got}")
print("\nMISMATCHES:", wrong if wrong else "none - config is as registered")

print("\nunconditional val (selection):", {k: round(v, 4) for k, v in result["final"].items()})
print("conditioned val (controls shown):",
      {k: round(v, 4) for k, v in (result.get("final_conditioned") or {}).items()})
print("dropped over max_len (train):", result.get("dropped_over_max_len"))
print("\nquick honouring check (n=200; NOT the registered C1 test):")
for label, cells in result.get("honoring_quick", {}).items():
    print(f"  {label:<10}" + "  ".join(f"{k} {100 * v:5.1f}%" for k, v in cells.items()))

# Recorded because E5's and R2's runs did not record them.
result["environment"] = {"tldextract": metadata.version("tldextract"),
                         "torch": torch.__version__,
                         "python": sys.version.split()[0],
                         "code_commit": manifest["commit"]}
print("\nenvironment:", result["environment"])

json.dump(result, open("/kaggle/working/e7_result.json", "w"), indent=1, default=str)
print("\nwrote /kaggle/working/e7_result.json")

---

## Download — the one thing that must not go wrong

The *Output* tab holds `best.pt`, `final.pt`, `train_result.json` and
`e7_result.json`. Download them (the zip is fine) and **rename the checkpoints
to `best_e7.pt` / `final_e7.pt`**. Never let a file called `best.pt` land over
anything in `data/`, and **never overwrite `best_e5.pt`**.

## What runs next, locally — nothing is adopted

In the order registered in `PREREG_E7.md`:

1. **C1 — controllability (primary).** Honouring at n = 4,000 per cell under the
   adopted decoder and delta. PASS needs >= 0.98 in all 8 cells.
2. **C2 — safety.** Gate v2 on E7's unconditional mode against gate v2 on E5.
   KILL if a verdict worsens *and* its x rises by more than 0.0141.
3. **C3 — consistency.** `<Q:1>` generation against the query-bearing rows of
   unconditional generation, same checkpoint, equal n.

Adoption is a separate decision that this notebook does not pre-authorise.